# Setup

In [ ]:
#FOR MAC, USE THIS BLOCK

import os, sys

SUMO_HOME  = r'C:\Program Files (x86)\Eclipse\Sumo'
SUMO_HOME  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.27.1/EclipseSUMO/share/sumo'
# PROJ_PATH  = '/Library/Frameworks/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/framework/EclipseSUMO.framework/Versions/1.22.0/EclipseSUMO/share/proj'

os.environ['SUMO_HOME']  = SUMO_HOME
# os.environ['PROJ_LIB']   = PROJ_PATH
# os.environ['PROJ_DATA']  = PROJ_PATH

sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# Verify all three
checks = {
    'SUMO_HOME':  os.path.exists(SUMO_HOME),
#     'proj.db':    os.path.exists(os.path.join(PROJ_PATH, 'proj.db')),
    'gtfs2pt.py': os.path.exists(f'{SUMO_HOME}/tools/import/gtfs/gtfs2pt.py'),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")

SUMO_HOME      : ❌ NOT FOUND
gtfs2pt.py     : ❌ NOT FOUND


In [2]:
#FOR WINDOWS, USE THIS BLOCK

import os, sys

# Edit this if SUMO is installed elsewhere (check with: where sumo)
SUMO_HOME = r'C:\Program Files (x86)\Eclipse\Sumo'
if not os.path.exists(os.path.join(SUMO_HOME, 'tools')):
    SUMO_HOME = r'C:\Program Files\Eclipse\Sumo'   # newer installs

# Optional: set this to the folder holding "gtfs data" and your .net.xml
# WORK_DIR = r'C:\Users\YOUR_NAME\path\to\DowntownSeattleSUMO\Simulation\GTFS'
# os.chdir(WORK_DIR)

os.environ['SUMO_HOME'] = SUMO_HOME
sys.path.append(os.path.join(SUMO_HOME, 'tools'))

# If you see "Cannot find proj.db" errors, uncomment and point at your proj folder:
# PROJ_PATH = os.path.join(SUMO_HOME, 'data', 'proj')
# os.environ['PROJ_LIB'] = PROJ_PATH
# os.environ['PROJ_DATA'] = PROJ_PATH

checks = {
    'SUMO_HOME': os.path.exists(SUMO_HOME),
    'gtfs2pt.py': os.path.exists(os.path.join(SUMO_HOME, 'tools', 'import', 'gtfs', 'gtfs2pt.py')),
}
for k, v in checks.items():
    print(f"{k:15s}: {'✅' if v else '❌ NOT FOUND'}")
print("cwd:", os.getcwd())

SUMO_HOME      : ✅
gtfs2pt.py     : ✅
cwd: c:\Users\dahh2\UW Projects\Seattle_Sumo\DowntownSeattleSUMO\Simulation\GTFS


In [3]:
# cd /Users/ziliqu/Dev/SDOT_Worldcup/Sumo_Test/2
# !cd "C:\Users\Soheil99\0 codes\DowntownSeattleSUMO\Simulation\GTFS"
# !cd


os.getcwd()

'c:\\Users\\dahh2\\UW Projects\\Seattle_Sumo\\DowntownSeattleSUMO\\Simulation\\GTFS'

# GTFS

In [4]:
import pandas as pd
import zipfile
import os

def load_gtfs(zip_path):
    data = {}
    with zipfile.ZipFile(zip_path, 'r') as z:
        for file in z.namelist():
            if file.endswith(".txt"):
                data[file] = pd.read_csv(z.open(file))
    return data


### Filtering to major Seattle GTFS Bus Lines

In [ ]:

# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/kcm_google_transit.zip"
OUTPUT_ZIP = "gtfs data/kcm_google_transit_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = ["24", "33", "70", "8", "C Line"]

# KEEP_ROUTES = [
#     "1", "2", "3", "4", "7", "8", "40",
#     "101", "120", "C Line", "D Line", "E Line", "H Line"
# ]

# Downtown Seattle bounding box
LAT_MIN, LAT_MAX = 47.58, 47.65
LON_MIN, LON_MAX = -122.37, -122.30

# ==============================
# LOAD GTFS FILES
# ==============================



gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]
## TODO ---soheil comment:
# do we have trips where the removed stops are in mid trips? trips that leave simulation area and then come back to it?
print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy every other untouched file as-is (shapes.txt, calendar.txt, calendar_dates.txt,
    # agency.txt, etc.) instead of a hardcoded allowlist 
    for fname in gtfs:
        if fname not in ['routes.txt', 'trips.txt', 'stop_times.txt', 'stops.txt']:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 4
Kept trips: 1270
Kept stops in downtown: 649
✅ Filtered GTFS saved to: gtfs data/kcm_google_transit_downtown.zip


In [6]:
# busgtfs = load_gtfs('gtfs data/kcm_google_transit_downtown.zip')
# stops = busgtfs['stops.txt']
# stops.stop_name.unique()
# stops.head()

In [7]:
# railgtfs = load_gtfs('gtfs data/rail_gtfs.zip')
# stops = railgtfs['stops.txt']
# # stops.stop_name.unique()
# # stops.head()
# # stops[stops["stop_name"].str.contains("Lynnwood", case=False, na=False)]
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126


# filtered_stops = stops[
#     (stops["stop_lat"] >= LAT_MIN) &
#     (stops["stop_lat"] <= LAT_MAX) &
#     (stops["stop_lon"] >= LON_MIN) &
#     (stops["stop_lon"] <= LON_MAX)
# ]

# filtered_stops

In [6]:
# ==============================
# CONFIG (EDIT THESE)
# ==============================

INPUT_ZIP = "gtfs data/rail_gtfs.zip"
OUTPUT_ZIP = "gtfs data/rail_gtfs_downtown.zip"

# Select important downtown routes 
KEEP_ROUTES = [
    "1 Line", "2 Line"
]

# Downtown Seattle bounding box
# #old ones
LAT_MIN, LAT_MAX = 47.31, 47.84
LON_MIN, LON_MAX = -122.43, -122.09

# new ones
# LAT_MIN, LAT_MAX = 47.5798124250899, 47.65143948560587
# LON_MIN, LON_MAX = -122.38544987028668, -122.29961918065126
# 47.5798124250899, -122.32752854299258 (below SODO)
# 47.65143948560587, -122.30397070065698 (above UW)
# 47.59038693450619, -122.29961918065126 (judkins park)
# 47.59525640734678, -122.38544987028668 (west seattle)

# ==============================
# LOAD GTFS FILES
# ==============================

gtfs = load_gtfs(INPUT_ZIP)

routes = gtfs["routes.txt"]
trips = gtfs["trips.txt"]
stop_times = gtfs["stop_times.txt"]
stops = gtfs["stops.txt"]

# ==============================
# STEP 1: FILTER ROUTES
# ==============================

filtered_routes = routes[routes["route_short_name"].isin(KEEP_ROUTES)]

print(f"Kept routes: {len(filtered_routes)}")

# ==============================
# STEP 2: FILTER TRIPS
# ==============================

filtered_trips = trips[trips["route_id"].isin(filtered_routes["route_id"])]

print(f"Kept trips: {len(filtered_trips)}")

# ==============================
# STEP 3: FILTER STOP TIMES
# ==============================

filtered_stop_times = stop_times[
    stop_times["trip_id"].isin(filtered_trips["trip_id"])
]

# ==============================
# STEP 4: FILTER STOPS (GEOGRAPHIC)
# ==============================

filtered_stops = stops[
    (stops["stop_lat"] >= LAT_MIN) &
    (stops["stop_lat"] <= LAT_MAX) &
    (stops["stop_lon"] >= LON_MIN) &
    (stops["stop_lon"] <= LON_MAX)
]

print(f"Kept stops in downtown: {len(filtered_stops)}")

# Keep only stop_times that use those stops
filtered_stop_times = filtered_stop_times[
    filtered_stop_times["stop_id"].isin(filtered_stops["stop_id"])
]

# ==============================
# STEP 5: CLEAN TRIPS (remove empty)
# ==============================

valid_trip_ids = filtered_stop_times["trip_id"].unique()
filtered_trips = filtered_trips[filtered_trips["trip_id"].isin(valid_trip_ids)]

# ==============================
# STEP 6: WRITE NEW GTFS ZIP
# ==============================

with zipfile.ZipFile(OUTPUT_ZIP, 'w') as z:
    def write(df, name):
        z.writestr(name, df.to_csv(index=False))

    write(filtered_routes, "routes.txt")
    write(filtered_trips, "trips.txt")
    write(filtered_stop_times, "stop_times.txt")
    write(filtered_stops, "stops.txt")

    # Copy unchanged files if they exist
#     for fname in ["calendar.txt", "calendar_dates.txt", "agency.txt"]:
#         if fname in gtfs:
#             write(gtfs[fname], fname)
    for fname in gtfs:
        if fname not in ['routes.txt', 'trips.txt', 'stop_times.txt', 'stops.txt']:
            write(gtfs[fname], fname)

print(f"✅ Filtered GTFS saved to: {OUTPUT_ZIP}")


Kept routes: 2
Kept trips: 17803
Kept stops in downtown: 309
✅ Filtered GTFS saved to: gtfs data/rail_gtfs_downtown.zip


### Adding GTFS to Sumo Network

#### suggestion: make sure previous outputs are deleted (fcd and resources folders)

In [7]:
import re


def dedupe_pt_stops(stops_file, vehicles_file):
    """Remove duplicate stop ids that gtfs2pt writes with --use-gtfs-stopids.

    gtfs2pt can map the same GTFS stop onto different lanes for different
    routes and writes one <busStop>/<trainStop> per mapping, all with the same id
    (sumo then fails with "Could not build busStop ...; probably declared twice").
    - same id + same lane  -> extra copies are dropped
    - same id + other lane -> renamed to <id>_1, <id>_2, ... and every
      <stop> in the vehicles file is pointed at the copy that lies on its route;
      renamed copies that no route uses are dropped
    Both files are rewritten in place.
    """
    stop_start = re.compile(r'^\s*<(busStop|trainStop)\b')
    attr = lambda name, text: re.search(r'\b%s="([^"]*)"' % name, text).group(1)
    lane_edge = lambda lane: lane.rsplit('_', 1)[0]

    with open(stops_file, encoding='utf-8') as f:
        lines = f.readlines()

    out = []
    variants = {}   # original id -> {edge: new id}
    stop_edge = {}  # final id -> edge
    dropped = renamed = 0
    i = 0
    while i < len(lines):
        m = stop_start.match(lines[i])
        if not m:
            out.append((None, [lines[i]]))
            i += 1
            continue
        # a stop element may span several lines (e.g. <access> children)
        j = i
        if not lines[i].rstrip().endswith('/>'):
            while '</%s>' % m.group(1) not in lines[j]:
                j += 1
        block = lines[i:j + 1]
        i = j + 1

        sid, edge = attr('id', block[0]), lane_edge(attr('lane', block[0]))
        seen = variants.setdefault(sid, {})
        if edge in seen:
            dropped += 1
            continue
        new_id = sid if not seen else '%s_%d' % (sid, len(seen))
        if new_id != sid:
            block[0] = block[0].replace('id="%s"' % sid, 'id="%s"' % new_id, 1)
            renamed += 1
        seen[edge] = new_id
        stop_edge[new_id] = edge
        out.append((new_id, block))

    multi = {sid: v for sid, v in variants.items() if len(v) > 1}
    retargeted = unmatched = unused = 0
    used = set()
    if multi:
        with open(vehicles_file, encoding='utf-8') as f:
            vlines = f.readlines()
        stop_ref = re.compile(r'\b(busStop|trainStop)="([^"]*)"')
        edges, pos = [], 0
        for k, line in enumerate(vlines):
            if re.search(r'<route\b', line) and 'edges="' in line:
                edges, pos = attr('edges', line).split(), 0
            m = stop_ref.search(line)
            if not m or '<stop' not in line:
                continue
            sid = m.group(2)
            # walk along the route so a route passing both edges picks the right one
            candidates = multi.get(sid, {stop_edge.get(sid): sid})
            best = None
            for edge, cand in candidates.items():
                if edge in edges[pos:]:
                    idx = edges.index(edge, pos)
                    if best is None or idx < best[0]:
                        best = (idx, cand)
            if best is None:
                if sid in multi:
                    unmatched += 1
                continue
            pos = best[0]
            used.add(best[1])
            if best[1] != sid:
                vlines[k] = line[:m.start(2)] + best[1] + line[m.end(2):]
                retargeted += 1
        with open(vehicles_file, 'w', encoding='utf-8') as f:
            f.writelines(vlines)

    # renamed copies that no route ended up using are just clutter
    renamed_ids = {v for vs in multi.values() for v in vs.values()} - set(multi)
    with open(stops_file, 'w', encoding='utf-8') as f:
        for sid, block in out:
            if sid in renamed_ids and sid not in used:
                unused += 1
                continue
            f.writelines(block)

    print(f"{stops_file}: dropped {dropped} duplicate stop(s), renamed {renamed}, "
          f"removed {unused} unused copy(ies); {vehicles_file}: retargeted {retargeted} stop reference(s)"
          + (f", {unmatched} could not be matched to a route edge" if unmatched else ""))

In [8]:
# !python $SUMO_HOME/tools/import/gtfs/gtfs2pt.py \
# -n merged.net.xml \
# --gtfs rail_gtfs_fil.zip \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose

# !python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
# -n merged_v2.net.xml \
# --gtfs "gtfs data/rail_gtfs_downtown.zip" \
# --date 20260329 \
# --modes=tram \
# --repair \
# --verbose

# outputs (routes, stops, vtypes) go to rail/ instead of the GTFS root
os.makedirs("rail", exist_ok=True)

!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n soheil_seattle_merged_pedspeed.net.xml \
--gtfs "gtfs data/rail_gtfs_downtown.zip" \
--date 20260329 \
--modes=tram \
--route-output rail/gtfs_pt_vehicles.add.xml \
--additional-output rail/gtfs_pt_stops.add.xml \
--vtype-output rail/vtypes.xml \
--repair \
--verbose

# gtfs2pt can write the same stop id more than once -> clean it up
dedupe_pt_stops("rail/gtfs_pt_stops.add.xml", "rail/gtfs_pt_vehicles.add.xml")

Loading net
function import_gtfs called at Sat, 03 Oct 2026 21:10:47 +0000
Loading GTFS data "gtfs data/rail_gtfs_downtown.zip"
function import_gtfs finished after 2.771286 seconds
Writing fcd file "fcd\gtfs\tram.fcd.xml"
One or more coordinates are negative, some applications might need strictly positive values. To avoid this use the option --shift.
Reusing old resources\gtfs\numerical.net.xml
Reusing old resources\gtfs\tram.net.xml
mapping tram
mapping trace with 15 points ... (261 router calls)
mapping trace with 26 points ... (299 router calls)
mapping trace with 22 points ... (270 router calls)
mapping trace with 26 points ... (299 router calls)
mapping trace with 4 points ... (5 router calls)
mapping trace with 8 points ... (9 router calls)
mapping trace with 11 points ... (36 router calls)
mapping trace with 22 points ... (286 router calls)
mapping trace with 11 points ... (36 router calls)
mapping trace with 26 points ... (290 router calls)
mapping trace with 26 points ... (290

In [ ]:
# !python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
# -n soheil_seattle_merged.net.xml \
# --gtfs "gtfs data/kcm_google_transit_downtown.zip" \
# --date 20260617 \
# --modes=bus \
# --repair \
# --verbose

# use this option to avoid duplicate bus stop error -- the second options gives you readability
# outputs (routes, stops, vtypes) go to bus/ instead of the GTFS root
# bus/patched_stops.add.xml pins stops whose GTFS coordinate snaps to the wrong edge
# (e.g. 1559 belongs on Alaskan Way before the turn, but its coordinate is closer to Columbia St)
os.makedirs("bus", exist_ok=True)

!python "$SUMO_HOME/tools/import/gtfs/gtfs2pt.py" \
-n soheil_seattle_merged_pedspeed.net.xml \
--gtfs "gtfs data/kcm_google_transit_downtown.zip" \
--date 20260617 \
--modes=bus \
--route-output bus/gtfs_pt_vehicles.add.xml \
--additional-output bus/gtfs_pt_stops.add.xml \
--vtype-output bus/vtypes.xml \
--patched-stops bus/patched_stops.add.xml \
--use-gtfs-stopids \
--repair \
--verbose

# gtfs2pt can write the same stop id more than once -> clean it up
dedupe_pt_stops("bus/gtfs_pt_stops.add.xml", "bus/gtfs_pt_vehicles.add.xml")

In [ ]:

with open("tram.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="tram"
           vClass="rail"
           accel="1.2"
           decel="1.5"
           sigma="0.3"
           length="40"
           maxSpeed="25"
           guiShape="rail"/>
</additional>
""")


In [6]:
with open("bus.add.xml", "w") as f:
    f.write("""<additional>
    <vType id="bus"
           vClass="bus"
           accel="1.0"
           decel="4.5"
           sigma="0.5"
           length="12"
           maxSpeed="16.7"
           guiShape="bus"/>
</additional>
""")


In [12]:
import re

with open("bus/gtfs_pt_stops.add.xml") as f:
    content = f.read()

# replace trainStop with busStop
content = content.replace("trainStop", "busStop")

with open("gtfs_pt_stops_fixed.add.xml", "w") as f:
    f.write(content)


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a tram.add.xml,gtfs_pt_stops_fixed.add.xml,gtfs_pt_vehicles.add.xml


In [ ]:
!sumo-gui \
-n merged.net.xml \
-a bus.add.xml,gtfs_pt_stops.add.xml,gtfs_pt_vehicles.add.xml